# 💾 Notebook 2: Atomicity and Transactions

Transactions are the first tool for handling contention. They provide **atomicity** - a group of operations either all succeed or all fail together.

## Learning Objectives

By the end of this notebook, you'll understand:
- What transactions are and how they work
- The ACID properties (especially Atomicity and Isolation)
- Why transactions alone don't prevent race conditions
- Different isolation levels and their trade-offs

## 🔄 What is a Transaction?

A transaction groups multiple database operations into a single unit:

```sql
BEGIN TRANSACTION;

-- All these operations happen together
UPDATE accounts SET balance = balance - 100 WHERE user_id = 'alice';
UPDATE accounts SET balance = balance + 100 WHERE user_id = 'bob';

COMMIT;  -- Make changes permanent
-- or ROLLBACK;  -- Undo everything
```

Either BOTH updates happen, or NEITHER does. No partial state.

---

🔍 **Open Adminer** at http://localhost:8080 to watch the `accounts` table during transfers!

In [ ]:
import psycopg2
from concurrent.futures import ThreadPoolExecutor
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "contention_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_connection()
    print("✅ Connected to PostgreSQL")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")

## 💰 Example: Bank Transfer

A classic example where atomicity is critical: transferring money between accounts.

In [ ]:
def show_balances():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT user_id, balance FROM accounts ORDER BY user_id")
    accounts = cursor.fetchall()
    conn.close()
    
    print("💰 Account Balances")
    print("=" * 30)
    total = 0
    for user_id, balance in accounts:
        print(f"   {user_id}: ${balance:.2f}")
        total += float(balance)
    print(f"   ────────────────")
    print(f"   Total: ${total:.2f}")
    return total

def reset_balances():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE accounts SET balance = 1000 WHERE user_id = 'alice'")
    cursor.execute("UPDATE accounts SET balance = 500 WHERE user_id = 'bob'")
    cursor.execute("UPDATE accounts SET balance = 2000 WHERE user_id = 'charlie'")
    conn.commit()
    conn.close()

reset_balances()
show_balances()

In [ ]:
def transfer_without_transaction(from_user: str, to_user: str, amount: float):
    conn = get_connection()
    cursor = conn.cursor()
    
    cursor.execute(
        "UPDATE accounts SET balance = balance - %s WHERE user_id = %s",
        (amount, from_user)
    )
    conn.commit()
    
    raise Exception("Simulated crash after debit!")
    
    cursor.execute(
        "UPDATE accounts SET balance = balance + %s WHERE user_id = %s",
        (amount, to_user)
    )
    conn.commit()
    conn.close()

print("🔥 Transfer WITHOUT Transaction (with crash)")
print("=" * 50)
print()
print("Before transfer:")
reset_balances()
before_total = show_balances()

print("\nAttempting to transfer $100 from Alice to Bob...")
try:
    transfer_without_transaction("alice", "bob", 100)
except Exception as e:
    print(f"\n💥 CRASH: {e}")

print("\nAfter crash:")
after_total = show_balances()

print(f"\n❌ Money disappeared! Before: ${before_total:.2f}, After: ${after_total:.2f}")
print("   Alice was debited, but Bob was never credited!")

In [ ]:
def transfer_with_transaction(from_user: str, to_user: str, amount: float):
    conn = get_connection()
    cursor = conn.cursor()
    
    try:
        cursor.execute(
            "UPDATE accounts SET balance = balance - %s WHERE user_id = %s",
            (amount, from_user)
        )
        
        raise Exception("Simulated crash after debit!")
        
        cursor.execute(
            "UPDATE accounts SET balance = balance + %s WHERE user_id = %s",
            (amount, to_user)
        )
        
        conn.commit()
        
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

print("✅ Transfer WITH Transaction (with crash)")
print("=" * 50)
print()
print("Before transfer:")
reset_balances()
before_total = show_balances()

print("\nAttempting to transfer $100 from Alice to Bob...")
try:
    transfer_with_transaction("alice", "bob", 100)
except Exception as e:
    print(f"\n💥 CRASH: {e}")
    print("   Transaction was ROLLED BACK!")

print("\nAfter crash:")
after_total = show_balances()

print(f"\n✅ Money preserved! Before: ${before_total:.2f}, After: ${after_total:.2f}")
print("   The crash caused a rollback - no partial state!")

## 🔬 ACID Properties

Transactions provide ACID guarantees:

| Property | Meaning | Example |
|----------|---------|--------|
| **A**tomicity | All or nothing | Transfer: both debit AND credit, or neither |
| **C**onsistency | Valid state to valid state | Balance can't go negative (if constrained) |
| **I**solation | Transactions don't interfere | Alice's transfer doesn't see Bob's half-done transfer |
| **D**urability | Committed = permanent | Power failure after COMMIT won't lose data |

## ⚠️ The Isolation Problem

Here's the crucial insight: **Atomicity alone doesn't prevent race conditions!**

Even with transactions, two transactions can still read the same initial state before either commits.

In [ ]:
def reset_concert():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE concerts SET available_seats = 1 WHERE id = 1")
    conn.commit()
    conn.close()

def buy_ticket_with_transaction(user_id: str):
    conn = get_connection()
    cursor = conn.cursor()
    
    try:
        cursor.execute("SELECT available_seats FROM concerts WHERE id = 1")
        seats = cursor.fetchone()[0]
        
        time.sleep(0.01)
        
        if seats >= 1:
            cursor.execute(
                "UPDATE concerts SET available_seats = available_seats - 1 WHERE id = 1"
            )
            cursor.execute(
                "INSERT INTO tickets (concert_id, user_id, seat_number, purchase_price) "
                "VALUES (1, %s, 'A1', 150.00)",
                (user_id,)
            )
            conn.commit()
            return True
        else:
            conn.rollback()
            return False
            
    except Exception as e:
        conn.rollback()
        return False
    finally:
        conn.close()

print("🎫 Testing Transactions with Race Condition")
print("=" * 50)

race_condition_count = 0

for i in range(10):
    reset_concert()
    
    with ThreadPoolExecutor(max_workers=2) as executor:
        f1 = executor.submit(buy_ticket_with_transaction, "alice")
        f2 = executor.submit(buy_ticket_with_transaction, "bob")
        
        alice = f1.result()
        bob = f2.result()
    
    if alice and bob:
        race_condition_count += 1
        print(f"   Test {i+1}: 🔥 Both bought (race condition!)")
    else:
        print(f"   Test {i+1}: ✅ Only one bought")

print(f"\n📊 Race conditions: {race_condition_count}/10")
if race_condition_count > 0:
    print("\n💡 Transactions provide atomicity but don't prevent race conditions!")
    print("   We need something more: LOCKING or ISOLATION LEVELS")

## 🔒 Isolation Levels

Databases offer different **isolation levels** that control how much transactions can see of each other:

| Level | Dirty Reads | Non-Repeatable Reads | Phantom Reads | Performance |
|-------|-------------|---------------------|---------------|-------------|
| READ UNCOMMITTED | ✅ Allowed | ✅ Allowed | ✅ Allowed | Fastest |
| READ COMMITTED | ❌ Prevented | ✅ Allowed | ✅ Allowed | Fast |
| REPEATABLE READ | ❌ Prevented | ❌ Prevented | ✅ Allowed | Medium |
| SERIALIZABLE | ❌ Prevented | ❌ Prevented | ❌ Prevented | Slowest |

PostgreSQL default: **READ COMMITTED**

In [ ]:
def buy_ticket_serializable(user_id: str):
    conn = get_connection()
    conn.set_isolation_level(psycopg2.extensions.ISOLATION_LEVEL_SERIALIZABLE)
    cursor = conn.cursor()
    
    try:
        cursor.execute("SELECT available_seats FROM concerts WHERE id = 1")
        seats = cursor.fetchone()[0]
        
        time.sleep(0.01)
        
        if seats >= 1:
            cursor.execute(
                "UPDATE concerts SET available_seats = available_seats - 1 WHERE id = 1"
            )
            cursor.execute(
                "INSERT INTO tickets (concert_id, user_id, seat_number, purchase_price) "
                "VALUES (1, %s, 'A1', 150.00)",
                (user_id,)
            )
            conn.commit()
            return {"success": True, "error": None}
        else:
            conn.rollback()
            return {"success": False, "error": "sold_out"}
            
    except psycopg2.errors.SerializationFailure as e:
        conn.rollback()
        return {"success": False, "error": "serialization_failure"}
    except Exception as e:
        conn.rollback()
        return {"success": False, "error": str(e)}
    finally:
        conn.close()

print("🔒 Testing SERIALIZABLE Isolation Level")
print("=" * 50)

results = {"both_success": 0, "one_success": 0, "serialization_error": 0}

for i in range(10):
    reset_concert()
    
    with ThreadPoolExecutor(max_workers=2) as executor:
        f1 = executor.submit(buy_ticket_serializable, "alice")
        f2 = executor.submit(buy_ticket_serializable, "bob")
        
        r1 = f1.result()
        r2 = f2.result()
    
    alice_success = r1["success"]
    bob_success = r2["success"]
    
    if alice_success and bob_success:
        results["both_success"] += 1
        status = "🔥 Both succeeded (shouldn't happen!)"
    elif r1["error"] == "serialization_failure" or r2["error"] == "serialization_failure":
        results["serialization_error"] += 1
        results["one_success"] += 1
        status = "✅ Serialization conflict detected!"
    else:
        results["one_success"] += 1
        status = "✅ Only one succeeded"
    
    print(f"   Test {i+1}: {status}")

print(f"\n📊 Results:")
print(f"   Both succeeded (bad):     {results['both_success']}/10")
print(f"   One succeeded (correct):  {results['one_success']}/10")
print(f"   Serialization errors:     {results['serialization_error']}/10")
print()
print("💡 SERIALIZABLE isolation detects conflicts and aborts one transaction!")
print("   The aborted transaction can retry.")

## ⚖️ Isolation Level Trade-offs

SERIALIZABLE prevents race conditions but has costs:

1. **Performance overhead** - Database must track all reads/writes
2. **Aborted transactions** - Need retry logic
3. **Potential deadlocks** - Circular dependencies

For most applications, **explicit locking** (next notebook) is more efficient than SERIALIZABLE isolation.

In [ ]:
print("📊 When to Use Each Isolation Level")
print("=" * 60)
print()
print("READ COMMITTED (default):")
print("   ✅ Good for most read operations")
print("   ✅ Low overhead")
print("   ❌ Doesn't prevent race conditions for writes")
print()
print("REPEATABLE READ:")
print("   ✅ Consistent reads within a transaction")
print("   ✅ Good for reports that need consistent snapshot")
print("   ❌ Still allows race conditions for concurrent writes")
print()
print("SERIALIZABLE:")
print("   ✅ Prevents all race conditions")
print("   ❌ Highest overhead")
print("   ❌ Requires retry logic for aborted transactions")
print()
print("💡 For write contention, explicit locking is usually better!")

## 🧪 Quick Quiz

1. **What's the difference between Atomicity and Isolation?**

2. **Why doesn't a transaction with default isolation prevent race conditions?**

3. **What happens when SERIALIZABLE detects a conflict?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Atomicity vs Isolation:")
print("   - ATOMICITY: Operations within ONE transaction all succeed or fail")
print("   - ISOLATION: How much transactions can see of EACH OTHER")
print("   - Atomicity doesn't prevent two transactions from conflicting!")
print()
print("2. Default isolation (READ COMMITTED) allows:")
print("   - Two transactions to read the same value")
print("   - Both to make decisions based on that value")
print("   - Both to commit their changes")
print("   - Result: race condition!")
print()
print("3. SERIALIZABLE conflict detection:")
print("   - Database aborts one of the conflicting transactions")
print("   - Throws SerializationFailure error")
print("   - Application must catch and RETRY the operation")

## 📚 Summary

### What We Learned

1. **Transactions** group operations into atomic units
2. **Atomicity** ensures all-or-nothing execution
3. **Default isolation** doesn't prevent race conditions
4. **SERIALIZABLE** detects conflicts but has overhead
5. **Explicit locking** is often more efficient

### Key Insight

> Transactions provide atomicity WITHIN themselves, but don't automatically prevent concurrent transactions from conflicting. We need additional mechanisms: isolation levels OR explicit locking.

### Next Up: Pessimistic Locking

In the next notebook, we'll learn how **SELECT ... FOR UPDATE** provides explicit control over concurrent access!